# IDS anomaly detection: train, validate, explain

Production-like учебный pipeline: фиксированные train/validation/test splits, fit Isolation Forest только на normal training rows, выбор threshold только на validation, финальная оценка на test и SHAP-анализ.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from scipy.stats import ks_2samp
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

from ids_ml_lab.features import FEATURES
from ids_ml_lab.modeling import (
    anomaly_scores,
    classification_metrics,
    make_pipeline,
    select_threshold,
)
from ids_ml_lab.training import bootstrap_ci, cross_validate, train

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data/prepared"
ARTIFACT = ROOT / "models/ids_iforest_v1"
sns.set_theme(style="whitegrid")

## 1. Фиксированный train/validation/test split

Splits подготовлены reproducible builder-ом с seed 42 и stratification. Мы не делаем новый случайный split в notebook: так test set остаётся стабильным и не попадает в выбор threshold.

In [ ]:
manifest = json.loads((DATA / "dataset_info.json").read_text())
train_df = pd.read_parquet(DATA / "train.parquet")
validation_df = pd.read_parquet(DATA / "validation.parquet")
test_df = pd.read_parquet(DATA / "test.parquet")
assert all(
    set(FEATURES + ["label", "attack_type"]) == set(frame.columns)
    for frame in [train_df, validation_df, test_df]
)
pd.DataFrame(
    {
        name: frame["label"].value_counts().sort_index()
        for name, frame in {"train": train_df, "validation": validation_df, "test": test_df}.items()
    }
).T

## 2. Preprocessing и Isolation Forest

Pipeline применяет OneHotEncoder к protocol/service/flag и RobustScaler к числовым полям. `handle_unknown=ignore` обязателен для live traffic. Модель обучается только на normal training rows; labels не используются при fit.

In [ ]:
pipeline = make_pipeline(SEED)
normal_train = train_df.loc[train_df.label == 0, FEATURES]
pipeline.fit(normal_train)
validation_y = validation_df.label.to_numpy()
test_y = test_df.label.to_numpy()
validation_scores = anomaly_scores(pipeline, validation_df[FEATURES])
test_scores = anomaly_scores(pipeline, test_df[FEATURES])
len(normal_train), validation_scores.shape, test_scores.shape

## 3. ROC AUC и PR AUC

Для anomaly score большее значение означает более аномальное событие. PR AUC особенно важен при несбалансированных классах.

In [ ]:
roc_auc = roc_auc_score(test_y, test_scores)
pr_auc = average_precision_score(test_y, test_scores)
fpr_curve, tpr_curve, _ = roc_curve(test_y, test_scores)
precision_curve, recall_curve, _ = precision_recall_curve(test_y, test_scores)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fpr_curve, tpr_curve, label=f"ROC AUC={roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC")
axes[0].legend()
axes[1].plot(recall_curve, precision_curve, label=f"PR AUC={pr_auc:.3f}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision–Recall")
axes[1].legend()
plt.tight_layout()

## 4. Threshold selection на validation

Выбираем максимум F1 среди thresholds с validation FPR ≤ 5%. Test labels здесь не используются.

In [ ]:
threshold = select_threshold(validation_y, validation_scores, max_fpr=0.05)
validation_metrics = classification_metrics(validation_y, validation_scores, threshold)
test_metrics = classification_metrics(test_y, test_scores, threshold)
pd.DataFrame([validation_metrics | {"split": "validation"}, test_metrics | {"split": "test"}]).drop(
    columns="confusion_matrix"
).set_index("split").T

In [ ]:
test_predictions = (test_scores >= threshold).astype(int)
ConfusionMatrixDisplay.from_predictions(
    test_y, test_predictions, display_labels=["normal", "attack"], cmap="Blues"
)
plt.title("Test confusion matrix")
plt.show()

## 5. Cross-validation и bootstrap confidence intervals

В каждом fold модель заново fit-ится только на normal-части training fold. Bootstrap CI считается на неизменном test score.

In [ ]:
cv_report = cross_validate(train_df, folds=5, seed=SEED)
bootstrap_report = {
    "roc_auc": bootstrap_ci(test_y, test_scores, roc_auc_score, iterations=500, seed=SEED),
    "pr_auc": bootstrap_ci(
        test_y, test_scores, average_precision_score, iterations=500, seed=SEED + 1
    ),
}
cv_report, bootstrap_report

## 6. KS separation

Two-sample KS сравнивает distributions anomaly score для normal и attack. Большой statistic означает более сильное разделение, но сам по себе не заменяет operational metrics.

In [ ]:
ks = ks_2samp(test_scores[test_y == 0], test_scores[test_y == 1])
{"ks_statistic": ks.statistic, "pvalue": ks.pvalue}

## 7. SHAP-анализ

TreeExplainer работает на transformed features. Global mean |SHAP| показывает, какие transformed признаки сильнее влияют на raw Isolation Forest output; знак не следует интерпретировать как причинность.

In [ ]:
preprocessor = pipeline.named_steps["preprocessor"]
forest = pipeline.named_steps["model"]
explain_rows = validation_df[FEATURES].sample(n=min(200, len(validation_df)), random_state=SEED)
X_explain = preprocessor.transform(explain_rows)
transformed_names = preprocessor.get_feature_names_out()
explainer = shap.TreeExplainer(forest)
shap_values = explainer(X_explain, check_additivity=False)
shap.summary_plot(
    shap_values.values, X_explain, feature_names=transformed_names, show=False, max_display=20
)
plt.tight_layout()
plt.show()

In [ ]:
shap_importance = pd.DataFrame(
    {"feature": transformed_names, "mean_abs_shap": np.abs(shap_values.values).mean(axis=0)}
).sort_values("mean_abs_shap", ascending=False)
shap_importance.head(20)

## 8. Export artifact

Для экспорта вызываем тот же канонический training path, что и CLI `uv run ids-train`. Он создаёт согласованный полный artifact, включая validation report, metadata и SHAP importance.

In [ ]:
export_report = train(DATA, ARTIFACT, seed=SEED)
expected_files = {
    "model.joblib",
    "features.json",
    "threshold.json",
    "metadata.json",
    "validation_report.json",
    "shap_importance.csv",
}
assert expected_files == {path.name for path in ARTIFACT.iterdir() if path.is_file()}
assert export_report["test"] == test_metrics
ARTIFACT